# Testing Prediction Request Model Serving

Notebook ini digunakan untuk menguji endpoint TensorFlow Serving, baik di lokal maupun setelah dideploy ke cloud. Ganti `MODEL_URL` dengan URL cloud Railway/Heroku jika model sudah dideploy.


In [5]:
import base64
import json

import requests
import tensorflow as tf

MODEL_URL = "https://mlops-production-cf92.up.railway.app/v1/models/iris-model:predict"

def make_tf_example(sepal_length, sepal_width, petal_length, petal_width):
    example = tf.train.Example(
        features=tf.train.Features(
            feature={
                "sepal_length": tf.train.Feature(float_list=tf.train.FloatList(value=[sepal_length])),
                "sepal_width": tf.train.Feature(float_list=tf.train.FloatList(value=[sepal_width])),
                "petal_length": tf.train.Feature(float_list=tf.train.FloatList(value=[petal_length])),
                "petal_width": tf.train.Feature(float_list=tf.train.FloatList(value=[petal_width])),
            }
        )
    )
    return example.SerializeToString()


## Kirim Prediction Request

Contoh di bawah memakai data Iris setosa. Response model berisi probabilitas untuk tiga kelas dan `class_id` prediksi.


In [6]:
serialized_example = make_tf_example(5.1, 3.5, 1.4, 0.2)
payload = {
    "instances": [
        {"examples": {"b64": base64.b64encode(serialized_example).decode("utf-8")}}
    ]
}

response = requests.post(MODEL_URL, json=payload, timeout=30)
print("Status code:", response.status_code)
print(json.dumps(response.json(), indent=2))


Status code: 200
{
  "predictions": [
    {
      "class_id": 0,
      "probabilities": [
        0.991691232,
        0.00667690439,
        0.00163191045
      ]
    }
  ]
}
